# Brain_Scape — 04 Analysis

Run segmentation, voxel scoring, damage classification, and confidence assessment on a preprocessed brain scan.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Preprocessed Volume

In [ ]:
data_dir = Path('../data')
job_id = 'notebook-demo'

# Find the best available preprocessed scan
registered_path = data_dir / 'registered' / job_id / 'registered.nii.gz'
stripped_path = data_dir / 'processed' / job_id / 'stripped.nii.gz'
nii_files = sorted(data_dir.rglob('*.nii.gz'))

if registered_path.exists():
    scan_path = str(registered_path)
elif stripped_path.exists():
    scan_path = str(stripped_path)
elif nii_files:
    scan_path = str(nii_files[0])
else:
    scan_path = None

if scan_path:
    img = nib.load(scan_path)
    data = img.get_fdata()
    print(f'Loaded: {scan_path}')
    print(f'Shape: {data.shape}')
else:
    print('No scan data found. Run 01 and 02 notebooks first.')

## 2. Segmentation

nnU-Net v2 primary, intensity-threshold fallback. Produces a binary lesion mask.

In [ ]:
if scan_path:
    from analysis.segmentation.segmentor import BrainSegmentor

    output_dir = data_dir / 'outputs' / job_id
    output_dir.mkdir(parents=True, exist_ok=True)

    segmentor = BrainSegmentor()
    result = segmentor.segment(scan_path, str(output_dir / 'lesion_mask.nii.gz'))

    mask_path = result['mask_path']
    print(f'Segmentation mask: {mask_path}')
    print(f'Method used: {result.get("method", "unknown")}')
    print(f'Dice score: {result.get("dice_score", "N/A")}')

    # Visualize mask overlay
    mask_img = nib.load(mask_path)
    mask_data = mask_img.get_fdata()
    mid_z = data.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(data[:, :, mid_z].T, cmap='gray', origin='lower')
    axes[0].set_title('Original Slice')
    axes[1].imshow(mask_data[:, :, mask_data.shape[2]//2].T, cmap='hot', origin='lower')
    axes[1].set_title('Lesion Mask')
    axes[2].imshow(data[:, :, mid_z].T, cmap='gray', origin='lower', alpha=0.7)
    if mask_data.shape[2] > 0:
        overlay = np.ma.masked_where(mask_data[:, :, mask_data.shape[2]//2].T == 0,
                                      mask_data[:, :, mask_data.shape[2]//2].T)
        axes[2].imshow(overlay, cmap='hot', alpha=0.5, origin='lower')
    axes[2].set_title('Overlay')
    plt.tight_layout()
    plt.show()

## 3. Voxel Scoring

Convert binary mask to per-region severity scores using atlas parcellation.

In [ ]:
if scan_path:
    from analysis.segmentation.voxel_scorer import VoxelScorer

    scorer = VoxelScorer()
    region_scores = scorer.score(
        mask_path=mask_path,
        volume_path=scan_path,
        atlas_dir=str(data_dir / 'atlases'),
    )

    print(f'Regions scored: {len(region_scores)}')
    print('\nTop 10 regions by severity:')
    sorted_regions = sorted(region_scores, key=lambda x: x.get('mean_severity', 0), reverse=True)
    for r in sorted_regions[:10]:
        print(f'  {r["anatomical_name"]}: severity={r.get("mean_severity", 0):.3f}, '
              f'volume={r.get("volume_mm3", 0):.1f}mm³, '
              f'pct={r.get("volume_pct_of_region", 0):.1f}%')

## 4. Damage Classification

Map per-region severity scores to the 5-level color scale: BLUE / GREEN / YELLOW / ORANGE / RED.

In [ ]:
if scan_path:
    from analysis.classification.damage_classifier import DamageClassifier

    classifier = DamageClassifier()
    damage_summary = classifier.classify(region_scores)

    print(f'Classified {len(damage_summary)} regions:')
    print()

    # Group by severity
    from collections import Counter
    severity_counts = Counter(r['severity_label'] for r in damage_summary)

    for label in ['RED', 'ORANGE', 'YELLOW', 'GREEN', 'BLUE']:
        count = severity_counts.get(label, 0)
        if count > 0:
            regions = [r['anatomical_name'] for r in damage_summary if r['severity_label'] == label]
            print(f'  {label} ({count} regions): {", ".join(regions[:3])}{"..." if len(regions) > 3 else ""}')

    # Visualize severity distribution
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = {'RED': '#E74C3C', 'ORANGE': '#E67E22', 'YELLOW': '#F1C40F',
              'GREEN': '#27AE60', 'BLUE': '#4A90D9'}
    labels = [l for l in ['RED', 'ORANGE', 'YELLOW', 'GREEN', 'BLUE'] if severity_counts.get(l, 0) > 0]
    counts = [severity_counts.get(l, 0) for l in labels]
    bar_colors = [colors[l] for l in labels]

    ax.bar(labels, counts, color=bar_colors, edgecolor='black')
    ax.set_xlabel('Severity Level')
    ax.set_ylabel('Number of Regions')
    ax.set_title('Damage Severity Distribution')
    plt.tight_layout()
    plt.show()

## 5. Confidence Scoring

Three-component confidence model: ensemble agreement (50%) + scan quality (30%) + registration accuracy (20%).

In [ ]:
if scan_path:
    from analysis.classification.confidence_scorer import ConfidenceScorer

    scorer = ConfidenceScorer()

    # Compute confidence for each classified region
    confidence_results = scorer.score(
        damage_summary=damage_summary,
        scan_path=scan_path,
        mask_path=mask_path,
    )

    overall_confidence = confidence_results.get('overall_confidence', 0)
    print(f'Overall confidence: {overall_confidence:.3f}')
    print(f'  Ensemble agreement:  {confidence_results.get("ensemble_agreement", 0):.3f}')
    print(f'  Scan quality:        {confidence_results.get("scan_quality", 0):.3f}')
    print(f'  Registration quality: {confidence_results.get("registration_accuracy", 0):.3f}')

    # Clinical vs display filter
    clinical_regions = [r for r in confidence_results.get('regions', [])
                        if r.get('clinical_confidence', 0) >= 0.5]
    display_regions = [r for r in confidence_results.get('regions', [])
                       if r.get('display_confidence', 0) >= 0.3]
    print(f'\nClinical-grade regions (>= 0.5): {len(clinical_regions)}')
    print(f'Display-grade regions (>= 0.3):  {len(display_regions)}')

## 6. Analysis JSON Contract

Assemble the final analysis output — the single source of truth between analysis and LLM layers.

In [ ]:
if scan_path:
    analysis_output = {
        'schema_version': '1.0.0',
        'scan_id': job_id,
        'damage_summary': damage_summary,
        'overall_confidence': overall_confidence,
        'confidence_breakdown': {
            'ensemble_agreement': confidence_results.get('ensemble_agreement', 0),
            'scan_quality': confidence_results.get('scan_quality', 0),
            'registration_accuracy': confidence_results.get('registration_accuracy', 0),
        },
        'differential_diagnosis': [],
        'connectivity': {},
        'scan_metadata': {
            'shape': list(img.shape),
            'voxel_size': list(img.header.get_zooms()[:3]),
            'modality': 'MRI_T1',
        },
    }

    # Save analysis JSON
    analysis_path = str(output_dir / 'analysis.json')
    with open(analysis_path, 'w') as f:
        json.dump(analysis_output, f, indent=2)
    print(f'Analysis JSON saved: {analysis_path}')

    # Verify required fields
    required_fields = ['schema_version', 'scan_id', 'damage_summary', 'overall_confidence']
    for field in required_fields:
        assert field in analysis_output, f'Missing required field: {field}'
    print('All required fields present in analysis JSON contract.')

    print('\nProceed to 05_llm_pipeline.ipynb to generate reports and Q&A.')